In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.metrics import accuracy_score,classification_report
from sklearn.model_selection import GridSearchCV

In [2]:
df=pd.read_csv('Loan.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [22]:
df.isnull().sum()

Loan_ID                       0
Gender                       13
Married                       3
Dependents                   15
Education                     0
Self_Employed                32
ApplicantIncome               0
CoapplicantIncome             0
LoanAmount                   22
Loan_Amount_Term             14
Credit_History               50
Property_Area                 0
Loan_Status                   0
Total_Income                  0
Loan_to_Income_Ratio         22
Estimated_Monthly_Payment    36
dtype: int64

In [23]:
X=df.drop(columns=['Loan_Status', 'Loan_ID'])
y=df['Loan_Status'].map({'Y': 1, 'N': 0})


In [47]:
X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y, test_size=0.1)
numerical_cols = [
    'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 
    'Loan_Amount_Term', 'Credit_History', 'Total_Income', 
    'Loan_to_Income_Ratio', 'Estimated_Monthly_Payment'
]

categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']

In [48]:
numeric=Pipeline([
    ('impute',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

In [49]:
categorical=Pipeline([
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('ohe',OneHotEncoder(sparse_output=False,handle_unknown='ignore',drop='first'))
])

In [50]:
preprocessor=ColumnTransformer([
    ('Numerical',numeric,numerical_cols),
    ('Categorical',categorical,categorical_cols)
])

In [51]:
pipelines = {
    'LogisticRegression': Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'SGDClassifier': Pipeline([('prep', preprocessor), ('clf', SGDClassifier(random_state=42))]),
    'Random Forest Classifier': Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(random_state=42))]),
    'XGBoost': Pipeline([('prep', preprocessor), ('clf', XGBClassifier(eval_metric='logloss', random_state=42))]),
    'KNN': Pipeline([('prep', preprocessor), ('clf', KNeighborsClassifier())])
}


In [52]:
param_grids = {
    'LogisticRegression': {
        'clf__C': [0.01, 0.1, 1.0, 10.0],
        'clf__class_weight': ['balanced', None]
    },
    'SGDClassifier': {
        'clf__alpha': [0.0001, 0.001, 0.01],
        'clf__loss': ['hinge', 'log_loss'],
        'clf__class_weight': ['balanced']
    },
    'KNN': {
        'clf__n_neighbors': [3, 5, 7, 11, 15],
        'clf__weights': ['uniform', 'distance']
    },
    'Random Forest Classifier': {
        'clf__n_estimators': [50, 100, 150],
        'clf__max_depth': [5, 10, 15],
        'clf__min_samples_leaf': [1, 2, 4],
        'clf__class_weight': ['balanced']
    },
    'XGBoost': {
        'clf__n_estimators': [50, 100, 150],
        'clf__max_depth': [3, 5, 7],
        'clf__learning_rate': [0.01, 0.05, 0.1],
        'clf__scale_pos_weight': [1, 3, 5]
    }
}


In [53]:
from sklearn.metrics import classification_report, roc_auc_score

best_overall_score = 0
best_overall_model = None
best_model_name = ""
results = []

# Quick check on class distribution before running
print("Target Class Distribution:\n", y_train.value_counts(normalize=True))

for name in pipelines.keys():
    print(f"\nRunning GridSearchCV for {name}...")
    
    # DYNAMIC FIX: Inject class balancing if the model supports it
    # This prevents the model from just guessing the majority class
    if 'clf' in pipelines[name].named_steps:
        model_step = pipelines[name].named_steps['clf']
        if hasattr(model_step, 'class_weight'):
            model_step.set_params(class_weight='balanced')
    
    # Initialize Grid Search with ROC-AUC scoring
    grid_search = GridSearchCV(
        estimator=pipelines[name],
        param_grid=param_grids[name],
        cv=5,                 
        scoring='roc_auc',   # 👈 CHANGED: Optimizes for class separation, not raw accuracy
        n_jobs=-1             
    )
    
    grid_search.fit(X_train, y_train)
    
    results.append({
        'Model': name,
        'Best CV ROC-AUC': grid_search.best_score_, # Updated label
        'Best Params': grid_search.best_params_
    })
    
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = name

# --- Final Summary ---
import pandas as pd
print("\n=== ALL GRID SEARCH RESULTS ===")
print(pd.DataFrame(results).to_string(index=False))
print(f"\n🏆 WINNER (by ROC-AUC): {best_model_name} with a Score of {best_overall_score:.4f}")

# --- Diagnostic Validation ---
# Run your best model on the training set to see if it is still predicting just one class
if best_overall_model is not None:
    y_pred = best_overall_model.predict(X_train)
    print(f"\n=== CLASSIFICATION REPORT FOR {best_model_name} (Train Set) ===")
    print(classification_report(y_train, y_pred))


Target Class Distribution:
 Loan_Status
1    0.684783
0    0.315217
Name: proportion, dtype: float64

Running GridSearchCV for LogisticRegression...

Running GridSearchCV for SGDClassifier...

Running GridSearchCV for Random Forest Classifier...

Running GridSearchCV for XGBoost...

Running GridSearchCV for KNN...

=== ALL GRID SEARCH RESULTS ===
                   Model  Best CV ROC-AUC                                                                                                  Best Params
      LogisticRegression         0.759716                                                            {'clf__C': 10.0, 'clf__class_weight': 'balanced'}
           SGDClassifier         0.759126                                 {'clf__alpha': 0.001, 'clf__class_weight': 'balanced', 'clf__loss': 'hinge'}
Random Forest Classifier         0.778048 {'clf__class_weight': 'balanced', 'clf__max_depth': 15, 'clf__min_samples_leaf': 1, 'clf__n_estimators': 50}
                 XGBoost         0.769289      

In [59]:
pipe=Pipeline([
    ('pre',preprocessor),
    ('model',RandomForestClassifier(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=1,
        class_weight='balanced',   # Crucial for handling the imbalanced data
        random_state=42
    ))
    ])
pipe.fit(X_train,y_train)
y_pred=pipe.predict(X_test)
print("ACCURACY:",accuracy_score(y_test,y_pred))

ACCURACY: 0.8709677419354839


In [60]:
test_predictions=pipe.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
print("\nClassification Report:\n", classification_report(y_test, test_predictions))
print("---------------------------------------------------------------------------------------")
print(f"Final Test Set Accuracy for {best_model_name}: {test_accuracy:.4f}")



Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.56      0.71        18
           1       0.85      1.00      0.92        44

    accuracy                           0.87        62
   macro avg       0.92      0.78      0.82        62
weighted avg       0.89      0.87      0.86        62

---------------------------------------------------------------------------------------
Final Test Set Accuracy for Random Forest Classifier: 0.8710
